<a href="https://colab.research.google.com/github/nithinn103n-coder/Code_Alpha_1/blob/main/Simple_RAG_Handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Ask questions of real research papers built live, in plain language

**Runtime:** Google Colab (CPU is fine, no GPU needed)

---

### What is RAG, in one sentence?

An LLM only knows what it was trained on and can't read your documents. **RAG** (Retrieval Augmented Generation) fixes this with a simple trick: before asking the LLM a question, first **find the relevant passages** from your own documents and **hand them to the LLM as context**, so it answers using real, grounded information instead of guessing.

### The whole idea in one picture

```
Your question
     │
     ▼
1. Turn the question into a vector (a list of numbers capturing its meaning)
     │
     ▼
2. Find document passages whose vectors are closest to it  ← "retrieval"
     │
     ▼
3. Hand those passages + the question to the LLM           ← "augmented generation"
     │
     ▼
An answer grounded in your real documents, with citations
```

That's it. Everything below is just building this, one small piece at a time, using **real research papers** as our documents (fittingly, papers about Transformers, RAG, and AI agents the exact topics from today's talk).

We'll build the **simple version first** fully working, fully explainable and only after that works, look at a short **"Bonus: Making It Smarter"** section for anyone who wants to see the more advanced production tricks.



## Part 1: Setup

We need three kinds of tools:
- A way to talk to an LLM (we use **Groq** free, no credit card needed)
- A way to turn text into vectors ("embeddings")
- A way to store and search those vectors quickly

Run the next two cells once.


In [1]:

# Install everything we need. This takes about a minute.
!pip install -q sentence-transformers faiss-cpu pypdf groq arxiv requests
print("Installed.")


Installed.


In [2]:

import os
import re
import textwrap
import numpy as np
from getpass import getpass

import faiss                                    # fast vector search
from sentence_transformers import SentenceTransformer   # turns text into vectors
import arxiv                                     # download real papers
import requests
from pypdf import PdfReader                      # read PDF text
from groq import Groq                            # talk to the LLM

print("Imports ready.")


Imports ready.


In [3]:

# Paste a free Groq API key here (get one at https://console.groq.com/keys — no credit card needed).
# The input box below hides what you type.
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

client = Groq()
MODEL = "openai/gpt-oss-120b"

def ask_llm(prompt: str, system: str = "", max_tokens: int = 600) -> str:
    """
    Sends one message to the LLM and returns its reply as plain text.
    Every other cell in this notebook calls this ONE function whenever it needs the LLM to think —
    that's the only place the model is actually called.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=0,          # 0 = give the most likely answer, not a random/creative one
        messages=messages,
    )
    return response.choices[0].message.content

# quick test
print(ask_llm("Reply with exactly: LLM connection OK"))


Enter your GROQ_API_KEY: ··········
LLM connection OK



## Part 2: Get Some Real Documents

We'll download 3 real, famous AI papers directly from arXiv. We use their **exact paper IDs** (not a keyword search) so we're guaranteed to get the correct papers every time.


In [4]:

# The exact papers we want, by their official arXiv ID.
PAPERS = [
    {"id": "1706.03762", "label": "Attention Is All You Need (the original Transformer paper)"},
    {"id": "2005.11401", "label": "Retrieval-Augmented Generation (the original RAG paper)"},
    {"id": "2210.03629", "label": "ReAct (reasoning + acting in AI agents)"},
]

os.makedirs("papers", exist_ok=True)
paper_files = []   # will hold {id, title, path} for each downloaded paper

client_arxiv = arxiv.Client()
for p in PAPERS:
    search = arxiv.Search(id_list=[p["id"]])
    result = next(client_arxiv.results(search))

    filepath = f"papers/{result.get_short_id()}.pdf"
    pdf_bytes = requests.get(result.pdf_url, timeout=30).content
    with open(filepath, "wb") as f:
        f.write(pdf_bytes)

    paper_files.append({"id": result.get_short_id(), "title": result.title, "path": filepath})
    print(f"Downloaded: {result.title}")

print(f"\n {len(paper_files)} papers saved to ./papers/")


Downloaded: Attention Is All You Need
Downloaded: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Downloaded: ReAct: Synergizing Reasoning and Acting in Language Models

 3 papers saved to ./papers/



## Part 3: Turn PDFs into Text Chunks

An LLM can't usefully process a 15 page PDF all at once, and we don't want to search "whole documents" we want to search **small, specific passages**. So we:

1. Extract the raw text from each PDF.
2. Cut it into **chunks** of a few hundred words each, with a little overlap between consecutive chunks (so we don't accidentally cut an idea in half at a chunk boundary).

This is the same idea as flipping through a textbook and photocopying it into index-card-sized pieces you can flip through quickly.


In [5]:

def get_pdf_text(path: str) -> str:
    """Reads every page of a PDF and joins the text together."""
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    text = "\n".join(pages)
    text = re.sub(r"\s+", " ", text)   # collapse messy whitespace into single spaces
    return text

def split_into_chunks(text: str, chunk_size: int = 800, overlap: int = 100):
    """
    Cuts text into overlapping chunks of `chunk_size` characters.
    `overlap` characters are repeated at the start of the next chunk, so an idea that
    spans a chunk boundary doesn't get lost.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Build our full list of chunks, across all 3 papers
all_chunks = []   # each item: {"id": ..., "title": ..., "text": ...}

for paper in paper_files:
    full_text = get_pdf_text(paper["path"])
    text_chunks = split_into_chunks(full_text)
    for i, chunk_text in enumerate(text_chunks):
        all_chunks.append({
            "id": f"{paper['id']}_chunk{i}",
            "title": paper["title"],
            "text": chunk_text,
        })

print(f" Created {len(all_chunks)} chunks from {len(paper_files)} papers")
print("\nExample chunk:")
print(textwrap.fill(all_chunks[10]["text"][:400], 100))


 Created 314 chunks from 3 papers

Example chunk:
is auto-regressive [10], consuming the previously generated symbols as additional input when
generating the next. 2 Figure 1: The Transformer - model architecture. The Transformer follows this
overall architecture using stacked self-attention and point-wise, fully connected layers for both
the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder
and Decode



## Part 4: Turn Chunks into Vectors, and Store Them

Now the key idea of RAG: we convert every chunk of text into a **vector** a list of numbers that captures its *meaning*. Chunks about similar topics end up with similar vectors (mathematically: they point in a similar direction).

We use a small, free, local model called `all-MiniLM-L6-v2` to do this conversion. Then we store all the vectors in **FAISS**, a library built for very fast "find the closest vectors to this one" search.


In [6]:

print("Loading the embedding model (turns text into vectors)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Turn every chunk's text into a vector
chunk_texts = [c["text"] for c in all_chunks]
vectors = embed_model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)
vectors = np.array(vectors, dtype="float32")

print(f"\nEach chunk became a vector with {vectors.shape[1]} numbers.")
print(f"Example — first 8 numbers of chunk 0's vector: {vectors[0][:8]}")

# Store all vectors in a FAISS index for fast searching
index = faiss.IndexFlatIP(vectors.shape[1])   # IP = inner product; with normalized vectors this IS cosine similarity
index.add(vectors)

print(f"\n {index.ntotal} vectors stored and ready to search.")


Loading the embedding model (turns text into vectors)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Each chunk became a vector with 384 numbers.
Example — first 8 numbers of chunk 0's vector: [-0.08058061 -0.14095847  0.04957414 -0.02534225  0.04007466  0.06426248
 -0.0988457   0.00130918]

 314 vectors stored and ready to search.



## Part 5: Search: Find the Most Relevant Chunks

To answer a question, we:
1. Turn the **question** into a vector, the same way we did for the chunks.
2. Ask FAISS: "which stored vectors are closest to this one?"
3. Return those chunks they're our best guess at "the passages that are relevant to this question."

Try it below with no LLM involved yet just to see retrieval working on its own.


In [7]:

def find_relevant_chunks(question: str, top_k: int = 4):
    """Returns the top_k chunks whose meaning is closest to the question."""
    question_vector = embed_model.encode([question], normalize_embeddings=True)
    similarities, indices = index.search(np.array(question_vector, dtype="float32"), top_k)

    results = []
    for idx, score in zip(indices[0], similarities[0]):
        results.append({"chunk": all_chunks[idx], "similarity": float(score)})
    return results

# Try it
test_question = "What is positional encoding used for?"
matches = find_relevant_chunks(test_question)

print(f"Question: {test_question}\n")
for m in matches:
    print(f"[similarity = {m['similarity']:.3f}]  {m['chunk']['id']}  ({m['chunk']['title'][:40]}...)")
    print(textwrap.fill(m["chunk"]["text"][:220], 100))
    print()


Question: What is positional encoding used for?

[similarity = 0.448]  1706.03762v7_chunk12  (Attention Is All You Need...)
ion. We also modify the self-attention sub-layer in the decoder stack to prevent positions from
attending to subsequent positions. This masking, combined with fact that the output embeddings are
offset by one position, e

[similarity = 0.446]  1706.03762v7_chunk23  (Attention Is All You Need...)
d allow the model to easily learn to attend by relative positions, since for any fixed offset k, P
Epos+k can be represented as a linear function of P Epos. We also experimented with using learned
positional embeddings [

[similarity = 0.437]  1706.03762v7_chunk22  (Attention Is All You Need...)
nce. To this end, we add "positional encodings" to the input embeddings at the bottoms of the
encoder and decoder stacks. The positional encodings have the same dimension dmodel as the
embeddings, so that the two can be

[similarity = 0.388]  1706.03762v7_chunk7  (Attention Is All 


## Part 6: Generate a Grounded Answer

Now we combine retrieval with the LLM:

1. Find the relevant chunks (Part 5).
2. Paste them into a prompt as "context."
3. Ask the LLM to answer **using only that context**, and to cite which chunk each fact came from.
4. Tell it explicitly to say "I don't know" if the context doesn't actually contain the answer this is the single most important instruction for avoiding made-up answers.


In [8]:

ANSWER_INSTRUCTIONS = (
    "You are a helpful research assistant. Answer the question using ONLY the "
    "context provided below. For every fact you state, mention which chunk it came from, like [chunk_id]. "
    "If the context does not contain the answer, say so clearly instead of guessing."
)

def answer_question(question: str, top_k: int = 4):
    matches = find_relevant_chunks(question, top_k=top_k)

    # Build the context block the LLM will read
    context_text = ""
    for m in matches:
        c = m["chunk"]
        context_text += f"\n[{c['id']}] (from: {c['title']})\n{c['text']}\n"

    prompt = f"Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer:"
    answer = ask_llm(prompt, system=ANSWER_INSTRUCTIONS, max_tokens=400)
    return answer, matches

# Try the full pipeline end to end
question = "What is positional encoding used for?"
answer, sources = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:")
print(textwrap.fill(answer, 100))
print("\nSOURCES USED:", [m["chunk"]["id"] for m in sources])


QUESTION: What is positional encoding used for?

ANSWER:
Positional encoding is added to the token embeddings so that the model can know **where each token
occurs in the sequence**. By summing a same‑dimensional positional vector with each input embedding
at the bottom of the encoder and decoder stacks, the Transformer injects information about absolute
and relative positions, which lets the attention mechanism learn to attend by relative offsets and
to generalize to longer sequences than seen during training【1706.03762v7_chunk22】. The sinusoidal
form was chosen because it can represent any fixed offset as a linear function of the encoding and
may allow extrapolation to longer sequence lengths【1706.03762v7_chunk23】.

SOURCES USED: ['1706.03762v7_chunk12', '1706.03762v7_chunk23', '1706.03762v7_chunk22', '1706.03762v7_chunk7']



## Part 7: Try It Yourself

Run this cell with a few different questions and see how it does. Try:
- A question clearly answerable from the papers
- A question the papers definitely do **not** cover (it should say so, not make something up)


In [9]:
questions_to_try = [
    "Why is attention called 'scaled dot-product attention'?",
    "What is the ReAct pattern for AI agents?",
    "What is the capital of France?",
]

for q in questions_to_try:
    print("="*90)
    print("QUESTION:", q)
    answer, sources = answer_question(q)
    print("\nANSWER:")


    print(textwrap.fill(answer, 100))
    print("\nSOURCES:", [m["chunk"]["id"] for m in sources])
    print()

QUESTION: Why is attention called 'scaled dot-product attention'?

ANSWER:
It is called **“scaled dot‑product attention”** because the attention weights are obtained by taking
the dot products between queries and keys **and then scaling those dot products by the
factor \(1/\sqrt{d_k}\)** before applying the softmax. The scaling prevents the dot products from
becoming too large (which would push the softmax into regions with extremely small gradients) and
makes the operation more stable — hence the name “scaled” dot‑product
attention [1706.03762v7_chunk13]. The authors explicitly introduced the scaling for this reason: “To
counteract this effect, we scale the dot products by \(1/\sqrt{d_k}\)” [1706.03762v7_chunk15].

SOURCES: ['1706.03762v7_chunk14', '1706.03762v7_chunk15', '1706.03762v7_chunk13', '1706.03762v7_chunk19']

QUESTION: What is the ReAct pattern for AI agents?

ANSWER:
The ReAct pattern augments an agent’s action space so that, in addition to ordinary environment
actions \(A


## 🎉 That's a complete, working RAG system

In about 40 lines of real code, you built:
- Document ingestion (real PDFs)
- Chunking
- Embeddings (vectors)
- Vector search (FAISS)
- Grounded generation with citations

This is genuinely the same *pattern* used in production systems the difference between this and a commercial product is mostly scale and the extra refinements below, not a different idea.

